In [1]:
import gc
import pandas as pd
import numpy as np
from collections import defaultdict
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split

ratings = pd.read_csv(
    "../data/ratings.csv",
    usecols=["userId", "movieId", "rating"],
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32"}
)
movies = pd.read_csv("../data/movies.csv", dtype={"movieId": "int32"})

min_user_ratings = 20
min_movie_ratings = 20

user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

active_users = set(user_counts[user_counts >= min_user_ratings].index)
popular_movies = set(movie_counts[movie_counts >= min_movie_ratings].index)

del user_counts, movie_counts
gc.collect()

ratings['user_ok'] = ratings['userId'].map(lambda x: x in active_users)
filtered = ratings[ratings['user_ok']].drop(columns='user_ok')
filtered = filtered[filtered['movieId'].isin(popular_movies)]

del ratings, active_users, popular_movies
gc.collect()

print(f"Filtered: {len(filtered):,} ratings")

SAMPLE_SIZE = 1_000_000   
if len(filtered) > SAMPLE_SIZE:
    filtered = filtered.sample(n=SAMPLE_SIZE, random_state=42)
    gc.collect()

print(f"Sample size for training: {len(filtered):,}")

Filtered: 31,725,920 ratings
Sample size for training: 1,000,000


In [3]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(filtered[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

model = SVD(n_factors=100, random_state=42)
model.fit(trainset)
predictions = model.test(testset)

del data, trainset
gc.collect()

0

In [4]:
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    user_est_true = defaultdict(list)
    for pred in predictions:
        user_est_true[pred.uid].append((pred.est, pred.r_ui))

    precisions = {}
    recalls = {}

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        n_rec_k = sum((est >= threshold) for (est, _) in user_ratings[:k])
        n_rel_and_rec_k = sum(
            ((true_r >= threshold) and (est >= threshold))
            for (est, true_r) in user_ratings[:k]
        )

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0
        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return precisions, recalls

precisions, recalls = precision_recall_at_k(predictions, k=10, threshold=3.5)

avg_precision = sum(precisions.values()) / len(precisions)
avg_recall = sum(recalls.values()) / len(recalls)

print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")

Precision@10: 0.5958
Recall@10: 0.5845


In [5]:
popularity_ranked = filtered.groupby('movieId').size().sort_values(ascending=False)
top_popular_movies = set(popularity_ranked.head(10).index)

def baseline_precision_recall(testset, top_popular_movies, threshold=3.5):
    user_actual = defaultdict(list)
    for uid, iid, true_r in testset:
        user_actual[uid].append((iid, true_r))

    precisions = []
    recalls = []
    for uid, items in user_actual.items():
        relevant = set(iid for iid, r in items if r >= threshold)
        recommended = top_popular_movies

        if len(recommended) > 0:
            precisions.append(len(relevant & recommended) / len(recommended))
        if len(relevant) > 0:
            recalls.append(len(relevant & recommended) / len(relevant))

    return np.mean(precisions), np.mean(recalls)

baseline_prec, baseline_rec = baseline_precision_recall(testset, top_popular_movies)
print(f"Baseline (popularity) — Precision@10: {baseline_prec:.4f}, Recall@10: {baseline_rec:.4f}")
print(f"SVD model          — Precision@10: {avg_precision:.4f}, Recall@10: {avg_recall:.4f}")

Baseline (popularity) — Precision@10: 0.0051, Recall@10: 0.0461
SVD model          — Precision@10: 0.5958, Recall@10: 0.5845
